# 12 · A3 stage 3: climate through the alignment or through POWER numbers (C2); aligned space or a classifier per tag (C3)

Rules fixed in `PLAN.md` (A3, A3.2) before this notebook ran. Held-out states only (Minnesota, Georgia),
questions as written with the state name swapped, ≥ 50 true pixels.

- **C2** (questions with a climate condition): (A) the whole question through the aligned space (notebook 11's
  best encoder); (B) the question's land-and-distance wording, rendered from the main model's A2 plan (cached
  gpt-oss-120b replies) with the climate removed, through the aligned space, with pixels in POWER cells that meet
  the plan's climate conditions ranked first. Paired AP difference (B − A).
- **C3**: the aligned space (whole question) against one logistic regression per land class on AlphaEarth, run
  through the same plan (land probability × conditions met). Paired top-k F1 difference (k = true pixels).
  A failed plan (missing, invalid or wrong area) scores F1 0 and AP = chance, as in A2.

In [1]:
import json
import time

import numpy as np
import pandas as pd

from askmap.catalogue import area_classes
from askmap.config import PROJECT_ROOT, data_root, load_config
from askmap.layers import SOLAR_LATER, official_leaf_map
from askmap.plan import AreaContext, execute, validate
from askmap.power import add_derived, fetch_daily, pixel_cells
from askmap.tiles import area_tiles, pixel_lonlat
from askmap.universe import Universe, box_of

cfg = load_config(PROJECT_ROOT / "configs" / "feasibility.yaml")
Q = load_config(PROJECT_ROOT / "configs" / "queries.yaml")["queries"]
RW = load_config(PROJECT_ROOT / "configs" / "reworded.yaml")["reworded"]
ROOT = data_root()
RES = PROJECT_ROOT / "outputs" / "results"
FIG = PROJECT_ROOT / "outputs" / "figures"
CL, P, SC = cfg["classes"], cfg["power"], cfg["scoring"]
LEAVES, GROUPS = CL["leaves"], CL["groups"]
CATALOGUE = area_classes(cfg)
T_START = time.time()
cells = pd.read_parquet(ROOT / "power" / "universe_cells.parquet")
daily = {
    box: add_derived(fetch_daily(P, b, ROOT / "power" / f"universe_{box}_daily.nc"), P["low_wind_sun_pct"])
    for box, b in P["boxes"].items()
}
uni = Universe(cells=cells, daily=daily)
SEEN = ["north_dakota", "north_carolina"]
HELD = cfg["heldout_pools"]  # seen area -> held-out state
POOL_Q = {
    "north_dakota": "north_dakota",
    "north_carolina": "north_carolina",
    "minnesota": "north_dakota",
    "georgia": "north_carolina",
}  # pool -> area whose questions it answers
T = {}
for area in POOL_Q:
    for name, (tile, role) in area_tiles(cfg["areas"][area], cfg["grid"]).items():
        z = np.load(ROOT / "tiles" / f"{area}_{name}.npz")
        solar = z["solar"] if "solar" in z.files else None
        nwi = z["nwi"] if "nwi" in z.files else None
        valid = np.isfinite(z["ae"]).all(axis=0)
        if solar is not None:
            valid &= solar != SOLAR_LATER
        lon, lat = pixel_lonlat(tile)
        box = box_of(float(lon.mean()), float(lat.mean()), P["boxes"])
        iy, ix = pixel_cells(daily[box].lat.values, daily[box].lon.values, lat, lon)
        T[name] = {
            "area": area,
            "role": role,
            "valid": valid,
            "box": box,
            "iy": iy,
            "ix": ix,
            "cell": uni.index(box, iy, ix),
            "official": official_leaf_map(z["cdl"], LEAVES, CL, nwi=nwi, solar=solar),
        }
        if role == "test":
            T[name]["ae_map"] = np.load(ROOT / "maps" / f"ae_{name}.npz")["leaf"]
            T[name]["planted_map"] = np.load(ROOT / "maps" / f"planted_{name}.npz")["leaf"]
CTX, TILES = {}, {}
for area in POOL_Q:
    names = [n for n, t in T.items() if t["area"] == area]
    ds = daily[T[names[0]]["box"]]
    ac = np.zeros((ds.sizes["lat"], ds.sizes["lon"]), bool)
    for n in names:
        ac[T[n]["iy"], T[n]["ix"]] = True
    CTX[area] = AreaContext(
        leaves=LEAVES,
        groups=GROUPS,
        res_m=cfg["grid"]["res_m"],
        power=ds,
        area_cells=ac,
        pixel_cell={n: (T[n]["iy"], T[n]["ix"]) for n in names},
    )
    TILES[area] = {
        "train": [n for n in names if T[n]["role"] == "train"],
        "test": [n for n in names if T[n]["role"] == "test"],
    }


def answer(plan, pool, maps="official"):
    """Answer of a plan in a pool's test tiles, as one vector over their valid pixels (fixed tile order)."""
    tiles = TILES[pool]["test"]
    key = "official" if maps == "official" else f"{maps}_map"
    got = execute(plan, {n: T[n][key] for n in tiles}, CTX[pool])
    return np.concatenate([got[n][T[n]["valid"]] for n in tiles])


def question_text(q, wording, pool):
    text = q["question"] if wording != "reworded" else RW[q["id"]]
    if pool in ("minnesota", "georgia"):
        text = text.replace("North Dakota", "Minnesota").replace("North Carolina", "Georgia")
    return text


def n_extra(plan):
    return len(plan["near"]) + len(plan["climate"]) + int(plan["land"]["min_patch_ha"] is not None)


print(f"{len(T)} tiles loaded; pools {dict((a, TILES[a]['test']) for a in POOL_Q)}")

18 tiles loaded; pools {'north_dakota': ['Kidder', 'Foster', 'Eddy'], 'north_carolina': ['NC4', 'NC5', 'NC6'], 'minnesota': ['MN1', 'MN2', 'MN3'], 'georgia': ['GA1', 'GA2', 'GA3']}


In [2]:
import os

import torch
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import average_precision_score

from askmap.aether_us import Projector, climate_ok_tile, plan_scores_tile, similarities, text_vectors
from askmap.captions import render_plan
from askmap.models import training_sample

AU = load_config(PROJECT_ROOT / "configs" / "aether_us.yaml")
CAP = load_config(PROJECT_ROOT / "configs" / "captions.yaml")
MCFG = AU["model"]
device = torch.device("mps" if torch.backends.mps.is_available() else "cpu")
V11 = json.loads((RES / "11_verdicts.json").read_text())
BEST = V11["best_encoder"]
KIND = {"clip": "clip", "modern": "sentence_transformer"}[BEST]
ENC_NAME = AU["text_encoders"][BEST]
OUT = ROOT / "aether_us"
ck = torch.load(OUT / f"projector_{BEST}.pt", weights_only=False)
MU, SD = ck["mu"], ck["sd"]
TXT = np.load(OUT / f"text_{BEST}.npz")
TPOS = {s: i for i, s in enumerate(TXT["strings"])}
TVEC = TXT["vectors"].astype(np.float32)
model = Projector(len(MU), TVEC.shape[1], MCFG["layers"], MCFG["hidden"]).to(device)
model.load_state_dict(ck["state"])
SUMMARY = np.load(ROOT / "climate" / "enc_summary.npy")
HELDOUT = ["minnesota", "georgia"]
NB = 2000


def unit(v):
    return v / np.linalg.norm(v, axis=-1, keepdims=True)


def med_ci(v, seed=0):
    v = np.asarray(v, float)
    r = np.random.default_rng(seed)
    b = [np.median(v[r.integers(0, len(v), len(v))]) for _ in range(NB)]
    return float(np.median(v)), float(np.percentile(b, 2.5)), float(np.percentile(b, 97.5))


def topk_f1(truth, score):
    k = int(truth.sum())
    return float(truth[np.argpartition(-score, k)[:k]].mean())


print(f"best encoder from notebook 11: {BEST} ({ENC_NAME})")

best encoder from notebook 11: clip (openai/clip-vit-large-patch14)


## Held-out pixels, their projected vectors, questions and the main model's cached plans

In [3]:
HP = {}
for pool in HELDOUT:
    ae, cl = [], []
    for n in TILES[pool]["test"]:
        flat = np.flatnonzero(T[n]["valid"].ravel())
        z = np.load(ROOT / "tiles" / f"{T[n]['area']}_{n}.npz")["ae"]
        ae.append(z.reshape(64, -1)[:, flat].T.astype(np.float32))
        cl.append(T[n]["cell"].ravel()[flat])
    HP[pool] = {"ae": np.concatenate(ae), "cell": np.concatenate(cl)}
TRUTH = dict(np.load(ROOT / "maps" / "truth.npz"))
LLM_DIR = PROJECT_ROOT / "outputs" / "llm" / "main"
rows = []
for q in Q:
    pool = HELD[q["plan"]["area"]]
    n = len(HP[pool]["cell"])
    g = np.unpackbits(TRUTH[f"{q['id']}|{pool}"])[:n].astype(bool)
    if g.sum() < SC["min_truth"]:
        continue
    rec = json.loads((LLM_DIR / f"as_written_{pool}_{q['id']}.json").read_text())
    plan, err = rec["plan"], rec["error"]
    if plan is not None:
        try:
            validate(plan, CATALOGUE, (2001, 2025))
            if plan["area"] != pool:
                plan, err = None, "wrong area"
        except (ValueError, KeyError, TypeError) as e:
            plan, err = None, f"invalid plan: {e}"
    rows.append(
        {
            "id": q["id"],
            "pool": pool,
            "text": question_text(q, "as_written", pool),
            "truth": g,
            "has_climate": bool(q["plan"]["climate"]),
            "plan": plan,
            "plan_error": err,
        }
    )
QS = rows
print(
    f"{len(QS)} scored held-out questions; {sum(r['plan'] is None for r in QS)} without a usable plan; "
    f"{sum(r['has_climate'] for r in QS)} with a climate condition"
)

40 scored held-out questions; 1 without a usable plan; 23 with a climate condition


## Text vectors: whole questions (cached from notebook 11) and land-and-distance wordings (new)

In [4]:
for r in QS:
    r["land_text"] = render_plan(dict(r["plan"], climate=[]), CAP, None) if r["plan"] is not None else None
new = sorted({r["land_text"] for r in QS if r["land_text"] is not None and r["land_text"] not in TPOS})
NEWVEC = (
    dict(zip(new, text_vectors(new, ENC_NAME, KIND, device, os.environ["HF_HOME"]), strict=True))
    if new
    else {}
)


def tvec(s):
    return TVEC[TPOS[s]] if s in TPOS else NEWVEC[s]


pd.DataFrame(
    [
        {"id": r["id"], "pool": r["pool"], "question": r["text"], "land_text": r["land_text"]}
        for r in QS
        if r["has_climate"]
    ]
).head(12)

/Volumes/disco2/github/ask-the-map-demo/.venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


W1029 19:09:14.115000 53594 torch/distributed/elastic/multiprocessing/redirects.py:35] NOTE: Redirects are currently not supported in MacOs.


Loading weights:   0%|          | 0/590 [00:00<?, ?it/s]

Loading weights:  42%|████▏     | 250/590 [00:00<00:00, 2490.98it/s]

Loading weights: 100%|██████████| 590/590 [00:00<00:00, 4189.09it/s]

,id,pool,question,land_text
0,nd09,minnesota,How much cropland in the Minnesota study area ...,Show cropland
1,nd15,minnesota,"In the Minnesota study area, show grassland an...",Show grassland and pasture
2,nd16,minnesota,"In the Minnesota study area, show grassland an...",Show grassland and pasture
3,nd18,minnesota,"In the Minnesota study area, show soybean fiel...",Show soybean fields
4,nd19,minnesota,"In the Minnesota study area, show spring wheat...",Show spring wheat
5,nd20,minnesota,How much cropland in the Minnesota study area ...,Show cropland
6,nd21,minnesota,"In the Minnesota study area, show sunflower fi...",NaN
7,nd23,minnesota,"In the Minnesota study area, show cropland in ...",Show cropland
8,nd24,minnesota,"In the Minnesota study area, show corn fields ...",Show corn fields
9,nd25,minnesota,"In the Minnesota study area, show cropland whe...",Show cropland


## C3 classifiers: one logistic regression per land class on AlphaEarth (training tiles)

In [5]:
CLS = cfg["classifier"]
embs, labs, keeps = [], [], []
for area, acfg in cfg["areas"].items():
    for name, (_tile, role) in area_tiles(acfg, cfg["grid"]).items():
        if role != "train":
            continue
        z = np.load(ROOT / "tiles" / f"{area}_{name}.npz")
        solar = z["solar"] if "solar" in z.files else None
        nwi = z["nwi"] if "nwi" in z.files else None
        embs.append(z["ae"].astype(np.float32))
        labs.append(official_leaf_map(z["cdl"], LEAVES, CL, nwi=nwi, solar=solar))
        k = np.ones(labs[-1].shape, bool)
        if solar is not None:
            k &= solar != SOLAR_LATER
        keeps.append(k)
XS, YS = training_sample(embs, labs, keeps, len(LEAVES), CLS["per_class"], CLS["seed"])
del embs
crng = np.random.default_rng(CLS["seed"])
COEF = np.zeros((len(LEAVES), 64), np.float32)
ICPT = np.full(len(LEAVES), -50.0, np.float32)  # "other" is never a query class: probability ~0
for li, leaf in enumerate(LEAVES):
    if leaf == "other" or not (YS == li).any():
        continue
    pos = XS[YS == li]
    neg = XS[YS != li]
    neg = neg[crng.choice(len(neg), size=min(len(neg), CLS["per_class"]), replace=False)]
    lr = LogisticRegression(max_iter=2000).fit(np.r_[pos, neg], np.r_[np.ones(len(pos)), np.zeros(len(neg))])
    COEF[li], ICPT[li] = lr.coef_[0], lr.intercept_[0]
np.savez_compressed(OUT / "tag_classifiers.npz", coef=COEF, intercept=ICPT, leaves=np.array(LEAVES))
print("trained", int((ICPT > -50).sum()), "classifiers on", len(XS), "pixels")

trained 15 classifiers on 286005 pixels


## Score every question

In [6]:
def probs_tile(n):
    ae = np.load(ROOT / "tiles" / f"{T[n]['area']}_{n}.npz")["ae"].astype(np.float32)
    lin = np.einsum("lc,chw->lhw", COEF, np.nan_to_num(ae)) + ICPT[:, None, None]
    return 1 / (1 + np.exp(-lin))


PROBS = {n: probs_tile(n) for pool in HELDOUT for n in TILES[pool]["test"]}
out = []
for pool in HELDOUT:
    mine = [r for r in QS if r["pool"] == pool]
    texts = [r["text"] for r in mine] + [r["land_text"] for r in mine if r["land_text"] is not None]
    sims = similarities(
        model,
        HP[pool]["ae"],
        HP[pool]["cell"],
        SUMMARY,
        MU,
        SD,
        unit(np.array([tvec(t) for t in texts])),
        device,
    )
    row_of = {t: i for i, t in enumerate(texts)}
    for r in mine:
        r["sim_q"] = sims[row_of[r["text"]]]
        r["sim_land"] = sims[row_of[r["land_text"]]] if r["land_text"] is not None else None
    del sims
for r in QS:
    pool, g = r["pool"], r["truth"]
    a = r["sim_q"]
    row = {
        "id": r["id"],
        "pool": pool,
        "has_climate": r["has_climate"],
        "plan_ok": r["plan"] is not None,
        "truth_px": int(g.sum()),
        "prevalence": float(g.mean()),
        "AP_aligned": average_precision_score(g, a),
        "F1_aligned": topk_f1(g, a),
    }
    if r["plan"] is None:
        row.update(AP_numbers=row["prevalence"], AP_classifier=row["prevalence"], F1_classifier=0.0)
    else:
        tiles = TILES[pool]["test"]
        clim = np.concatenate([climate_ok_tile(r["plan"], n, CTX[pool])[T[n]["valid"]] for n in tiles])
        b = r["sim_land"] + 2.0 * clim
        c3 = np.concatenate(
            [plan_scores_tile(r["plan"], PROBS[n], n, CTX[pool])[T[n]["valid"]] for n in tiles]
        )
        row.update(
            AP_numbers=average_precision_score(g, b),
            AP_classifier=average_precision_score(g, c3),
            F1_classifier=topk_f1(g, c3),
        )
    out.append(row)
SC12 = pd.DataFrame(out)
tr05 = pd.read_csv(RES / "05_translation.csv")
ref = tr05[(tr05.arm == "main") & (tr05.wording == "as_written") & tr05.pool.isin(HELDOUT)][
    ["id", "pool", "F1_end_to_end"]
]
SC12 = SC12.merge(
    ref.rename(columns={"F1_end_to_end": "F1_A2_alphaearth_maps"}), on=["id", "pool"], how="left"
)
SC12.to_csv(RES / "12_scores.csv", index=False)
SC12.drop(columns=["truth_px"]).round(3)

,id,pool,has_climate,plan_ok,prevalence,AP_aligned,F1_aligned,AP_numbers,AP_classifier,F1_classifier,F1_A2_alphaearth_maps
0,nd01,minnesota,False,True,0.026,0.047,0.050,0.048,0.271,0.321,0.253
1,nd02,minnesota,False,True,0.003,0.005,0.000,0.005,0.085,0.115,0.085
2,nd03,minnesota,False,True,0.026,0.055,0.059,0.048,0.271,0.321,0.253
3,nd04,minnesota,False,True,0.226,0.597,0.670,0.822,0.780,0.742,0.838
4,nd05,minnesota,False,True,0.011,0.074,0.122,0.092,0.314,0.394,0.292
5,nd09,minnesota,True,True,0.347,0.881,0.908,0.972,0.978,0.928,0.935
6,nd15,minnesota,True,True,0.160,0.505,0.521,0.695,0.689,0.649,0.610
7,nd16,minnesota,True,True,0.067,0.214,0.264,0.706,0.662,0.620,0.611
8,nd18,minnesota,True,True,0.077,0.561,0.665,0.966,0.975,0.917,0.919
9,nd19,minnesota,True,True,0.000,0.085,0.124,0.103,0.068,0.021,0.223


## Verdicts C2 and C3

In [7]:
c2 = SC12[SC12.has_climate]
d2 = (c2.AP_numbers - c2.AP_aligned).to_numpy()
m2, lo2, hi2 = med_ci(d2)
C2 = {
    "n": len(c2),
    "median_diff_AP_numbers_minus_aligned": m2,
    "lo": lo2,
    "hi": hi2,
    "median_AP_aligned": float(c2.AP_aligned.median()),
    "median_AP_numbers": float(c2.AP_numbers.median()),
    "numbers_win": lo2 > 0,
}
d3 = (SC12.F1_classifier - SC12.F1_aligned).to_numpy()
m3, lo3, hi3 = med_ci(d3)
C3 = {
    "n": len(SC12),
    "median_diff_F1_classifier_minus_aligned": m3,
    "lo": lo3,
    "hi": hi3,
    "median_F1_aligned": float(SC12.F1_aligned.median()),
    "median_F1_classifier": float(SC12.F1_classifier.median()),
    "median_F1_A2_alphaearth_maps": float(SC12.F1_A2_alphaearth_maps.median()),
    "answer_key_F1": 1.0,
    "winner": "classifier per tag" if lo3 > 0 else ("aligned space" if hi3 < 0 else "tie"),
}
V = {"encoder": BEST, "C2": C2, "C3": C3}
(RES / "12_verdicts.json").write_text(json.dumps(V, indent=1))
print(json.dumps(V, indent=1))

{
 "encoder": "clip",
 "C2": {
  "n": 23,
  "median_diff_AP_numbers_minus_aligned": 0.25028275720532356,
  "lo": 0.1442375595943014,
  "hi": 0.42333306627848866,
  "median_AP_aligned": 0.21429415380383227,
  "median_AP_numbers": 0.6952270432460023,
  "numbers_win": true
 },
 "C3": {
  "n": 40,
  "median_diff_F1_classifier_minus_aligned": 0.1621443252463171,
  "lo": 0.07195368631114885,
  "hi": 0.25911682283361437,
  "median_F1_aligned": 0.1829685079122258,
  "median_F1_classifier": 0.5619548616534145,
  "median_F1_A2_alphaearth_maps": 0.48896011396011396,
  "answer_key_F1": 1.0,
  "winner": "classifier per tag"
 }
}


## Questions that worked and failed (classifier per tag and aligned space)

In [8]:
qtext = {r["id"]: r["text"] for r in QS}
show = SC12.assign(question=SC12.id.map(qtext))
cols = ["id", "pool", "question", "F1_aligned", "F1_classifier", "AP_aligned", "AP_numbers", "prevalence"]
pd.set_option("display.max_colwidth", 120)
pd.concat([show.nlargest(6, "F1_classifier"), show.nsmallest(6, "F1_classifier")])[cols].round(3)

,id,pool,question,F1_aligned,F1_classifier,AP_aligned,AP_numbers,prevalence
12,nd23,minnesota,"In the Minnesota study area, show cropland in the warmest half of the study area, by mean May-September temperature ...",0.794,0.941,0.832,0.976,0.251
10,nd20,minnesota,How much cropland in the Minnesota study area had a 2021 growing season (May-September mean temperature) among the h...,0.689,0.934,0.624,0.975,0.265
14,nd25,minnesota,"In the Minnesota study area, show cropland where May-September 2024 rainfall was in the wettest 20% of years from 20...",0.630,0.934,0.548,0.972,0.229
5,nd09,minnesota,How much cropland in the Minnesota study area got at least 250 mm of rain in April-June 2022?,0.908,0.928,0.881,0.972,0.347
13,nd24,minnesota,"In the Minnesota study area, show corn fields in the third of the study area with the most growing degree days (base...",0.597,0.923,0.531,0.977,0.116
8,nd18,minnesota,"In the Minnesota study area, show soybean fields where August 2023 rainfall was in the driest 20% of Augusts from 20...",0.665,0.917,0.561,0.966,0.077
11,nd21,minnesota,"In the Minnesota study area, show sunflower fields that had fewer than 1,300 growing degree days (base 10 C) from Ma...",0.000,0.000,0.003,0.000,0.000
15,nd26,minnesota,"In the Minnesota study area, show developed land in the colder half of the study area, by yearly heating degree days...",0.581,0.000,0.471,0.070,0.077
25,nc04,georgia,"In the Georgia study area, show patches of grassland and pasture of at least 10 ha within 2 km of an existing solar ...",0.000,0.000,0.000,0.003,0.000
9,nd19,minnesota,"In the Minnesota study area, show spring wheat where June-July 2021 rainfall was in the driest 10% since 2001 (throu...",0.124,0.021,0.085,0.103,0.000


In [9]:
print(f"Run time {(time.time() - T_START) / 60:.1f} min")

Run time 1.0 min
